In [ ]:
!pip install transformers datasets faiss-cpu sentence-transformers fastapi uvicorn nest-asyncio

In [ ]:
import re
from typing import Dict, Any, List


def normalize_text(value: str) -> str:
    return value.strip().lower()


def slugify(value: str) -> str:
    value = normalize_text(value)
    value = re.sub(r'[^a-z0-9]+', '_', value)
    return value.strip('_')


def clean_list(values: List[str]) -> List[str]:
    if not values:
        return []
    return list(set([normalize_text(v) for v in values if v and v.strip()]))


def extract_primary_location(locations: List[str]) -> str:
    if not locations:
        return ""
    # Take first location and simplify (e.g., "Gurgaon, Haryana" → "gurgaon")
    primary = locations[0].split(",")[0]
    return normalize_text(primary)


def generate_role_id(company: str, title: str, role_type: str) -> str:
    return f"{slugify(company)}_{slugify(title)}_{slugify(role_type)}"


def generate_embedding_text(data: Dict[str, Any]) -> str:
    role = data.get("role", {})
    company = data.get("company", {})
    skills = data.get("skills", {})
    eligibility = data.get("eligibility", {})
    selection = data.get("selection_process", [])

    title = role.get("title", "")
    company_name = company.get("name", "")
    category = role.get("category", "")
    role_type = role.get("type", "")
    location = ", ".join(role.get("location", [])[:1])

    degrees = ", ".join(eligibility.get("degrees", []))
    years = ", ".join(eligibility.get("year", []))

    must_have = ", ".join(skills.get("must_have", []))
    selection_steps = ", ".join(selection)

    parts = []

    # Sentence 1
    if title and company_name:
        parts.append(
            f"{title} role at {company_name}"
            + (f" for {years} {degrees} candidates" if years or degrees else "")
            + "."
        )

    # Sentence 2
    if must_have or category:
        parts.append(
            f"Requires skills in {must_have}"
            + (f" within {category}" if category else "")
            + "."
        )

    # Sentence 3
    if selection_steps:
        parts.append(
            f"The selection process includes {selection_steps}."
        )

    # Sentence 4 (optional context)
    if role_type or location:
        parts.append(
            f"This is a {role_type} role"
            + (f" based in {location}" if location else "")
            + "."
        )

    return " ".join(parts).strip()


def build_filters(data: Dict[str, Any]) -> Dict[str, Any]:
    role = data.get("role", {})
    skills = data.get("skills", {})

    return {
        "role": slugify(role.get("title", "")),
        "type": slugify(role.get("type", "")),
        "location": extract_primary_location(role.get("location", [])),
        "skills": clean_list(skills.get("must_have", [])),
        "difficulty": normalize_text(data.get("difficulty", ""))
    }


def preprocess(data: Dict[str, Any]) -> Dict[str, Any]:
    role = data.get("role", {})
    company = data.get("company", {})

    role_id = generate_role_id(
        company.get("name", ""),
        role.get("title", ""),
        role.get("type", "")
    )

    embedding_text = generate_embedding_text(data)
    filters = build_filters(data)

    # Clean metadata (optional: you can prune fields here)
    metadata = data.copy()
    metadata["role_id"] = role_id

    return {
        "id": role_id,
        "embedding_text": embedding_text,
        "filters": filters,
        "metadata": metadata
    }

In [ ]:
data = {
  "company": {
    "name": "TechNova Solutions",
    "industry": ["Software Development", "SaaS"],
    "tags": ["startup", "b2b", "cloud"]
  },
  "role": {
    "title": "Backend Developer",
    "category": "Software Development",
    "type": "Full-time",
    "location": ["Bangalore, Karnataka, India"],
    "mode": "Onsite"
  },
  "eligibility": {
    "degrees": ["BTech", "BE"],
    "branches": ["Computer Science", "IT"],
    "year": ["2025", "2026"]
  },
  "skills": {
    "must_have": ["Java", "Spring Boot", "DSA"],
    "good_to_have": ["Docker", "AWS", "Microservices"]
  },
  "selection_process": [
    "Online Assessment",
    "Technical Interview",
    "HR Interview"
  ],
  "difficulty": "Medium",
  "salary": {
    "min": 6,
    "max": 12,
    "currency": "LPA"
  },
  "tags": ["backend", "api", "cloud"],
  "source": "LinkedIn",
  "last_updated": "2026-04"
}
preprocess(data)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('BAAI/bge-large-en-v1.5')

embedding_vector = model.encode("Your preprocessed embedding_text here")
print(len(embedding_vector), embedding_vector[:])